# Env

In [ ]:
import os

import torch

from tokenizers import ByteLevelBPETokenizer  # CharBPETokenizer
from transformers import (AutoTokenizer,
                          T5TokenizerFast,
                          pipeline,
                          AutoModel,
                          AutoModelForSequenceClassification,
                          AutoModelForTokenClassification,
                          AutoModelForQuestionAnswering,
                          AutoModelForSeq2SeqLM,
                          AutoModelForCausalLM,
                          Trainer,
                          TrainingArguments)
from datasets import load_dataset

In [ ]:
# work dir
work_dir = '/home/ubuntu/nlp-practice'

In [ ]:
%cd {work_dir}
!pwd

In [ ]:
# tokeinzer warning disable
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Tokenizer

## Train Tokenizer

In [ ]:
# !python train_tokenizer.py

## Test Tokenizer

In [ ]:
# Tokenizer load
output_dir = "data/aihub_koen_32k"
tokenizer = T5TokenizerFast.from_pretrained(output_dir)

In [ ]:
# 동작 확인을 위한 문장
ko_sentence = "<s>동해 물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라 만세 무궁화 삼천리 화려강산 대한 사람 대한으로 길이 보전하세</s>"
en_sentence = "<s>May God protect us until the waters of the East Sea and Mt. Baekdu dry up and wear away, and may our country live forever as a splendid land of the Rose of Sharon and the Korean people.</s>"

In [ ]:
# 한국어 문장
ids = tokenizer.encode(ko_sentence)
tokens = tokenizer.tokenize(ko_sentence)

print(">>>", tokens)
print(">>>", ids)
print(">>>", tokenizer.decode(ids))

In [ ]:
# 영어 문장
ids = tokenizer.encode(en_sentence)
tokens = tokenizer.tokenize(en_sentence)

print(">>>", tokens)
print(">>>", ids)
print(">>>", tokenizer.decode(ids))

## Pre-trained Tokenizer

In [ ]:
# load pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

In [ ]:
# 한국어 문장
ids = tokenizer.encode(ko_sentence)
tokens = tokenizer.tokenize(ko_sentence)

print(">>>", tokens)
print(">>>", ids)
print(">>>", tokenizer.decode(ids))

In [ ]:
# 영어 문장
ids = tokenizer.encode(en_sentence)
tokens = tokenizer.tokenize(en_sentence)

print(">>>", tokens)
print(">>>", ids)
print(">>>", tokenizer.decode(ids))

## Padding with Tokenizer

In [ ]:
texts = [
    "동해물과 백두산이 마르고 닳도록 하느님이 보우하사 우리나라 만세",
    "무궁화 삼천리 화려 강산",
    "대한 사람 대한으로 길이 보전하세"
]

In [ ]:
# Tokenizer load
output_dir = "data/aihub_koen_32k"
tokenizer = T5TokenizerFast.from_pretrained(output_dir)
# tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

In [ ]:
# hugging face tokenizer
result = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)
result

# Embedding Layer

In [ ]:
texts = [
    "무궁화 삼천리 화려 강산",
    "대한 사람 대한으로 길이 보전하세"
]

In [ ]:
# hugging face tokenizer
output_dir = "data/aihub_koen_32k"
tokenizer = T5TokenizerFast.from_pretrained(output_dir)

result = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)
result

In [ ]:
# Embedding Layer
embedding = torch.nn.Embedding(
    tokenizer.vocab_size,
    4,
    padding_idx=tokenizer.pad_token_id
)

In [ ]:
embedding(result['input_ids'])

# Hugging Face 활용

## pipeline
- 참고: https://huggingface.co/docs/transformers/task_summary

In [ ]:
classifier = pipeline(task="sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
preds = classifier("Hugging Face is the best thing since sliced bread!")
preds

In [ ]:
classifier = pipeline(task="ner",
                      model="dbmdz/bert-large-cased-finetuned-conll03-english")
preds = classifier("Hugging Face is a French company based in New York City.")
preds

In [ ]:
question_answerer = pipeline(task="question-answering",
                             model="distilbert/distilbert-base-cased-distilled-squad")
preds = question_answerer(
    question="What is the name of the repository?",
    context="The name of the repository is huggingface/transformers",
)
preds

In [ ]:
summarizer = pipeline(task="summarization",
                      model="sshleifer/distilbart-cnn-12-6",
                      max_length=56)
result = summarizer(
    "In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In the former task our best model outperforms even all previously reported ensembles."
)
result

## AutoTokenizer

In [ ]:
sentence = "May God protect us until the waters of the East Sea and Mt. Baekdu dry up and wear away, and may our country live forever as a splendid land of the Rose of Sharon and the Korean people."

https://huggingface.co/klue/bert-base

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
tokens = tokenizer.tokenize(sentence)
print(tokens)

https://huggingface.co/google-bert/bert-base-cased

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")
tokens = tokenizer.tokenize(sentence)
print(tokens)

https://huggingface.co/openai-community/gpt2

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")
tokens = tokenizer.tokenize(sentence)
print(tokens)

## AutoModel
- 참고: https://huggingface.co/transformers/v3.0.2/model_doc/auto.html

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base",
                                                           num_labels=2)
model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained("klue/bert-base",
                                                        num_labels=2)
model

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained("klue/bert-base")
model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("KETI-AIR/ke-t5-base-ko")
model

In [ ]:
model = AutoModelForCausalLM.from_pretrained("skt/kogpt2-base-v2")
model

## Datasets

In [ ]:
class TutorialDataset(torch.utils.data.Dataset):
    def __init__(self):
        super().__init__()

        self.texts = [
            "강력 추천합니다.",
            "나중에 집에서 보는게 딱 좋은영화"
        ]
        self.labels = [1, 0]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

dataset = TutorialDataset()
print(dataset[1])

In [ ]:
dataset = load_dataset("e9t/nsmc")
print(dataset)
print(dataset['train'][100])

In [ ]:
dataset = load_dataset("KorQuAD/squad_kor_v1")
print(dataset)
print(dataset['train'][100])

In [ ]:
dataset = load_dataset("HSJuan/aihub-ko-en-literary")
print(dataset)
print(dataset['train'][100])

## Trainer, TrainingArguments

In [ ]:
dataset = load_dataset("e9t/nsmc")
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base",
                                                           num_labels=2)

In [ ]:
class TutorialDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        super().__init__()

        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        return row["document"], row["label"]

In [ ]:
train_dataset = TutorialDataset(dataset['train'].select(range(100)))
train_dataset[10]

In [ ]:
test_dataset = TutorialDataset(dataset['test'].select(range(50)))
test_dataset[10]

In [ ]:
class TutorialCollator:
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        documents, labels = zip(*batch)

        inputs = self.tokenizer(
            documents,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": torch.tensor(labels, dtype=torch.long),
        }

In [ ]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./results/hf-tutorial",      # 결과 저장 경로
    num_train_epochs=3,                      # 학습 횟수
    per_device_train_batch_size=8,           # 학습 배치 크기
    per_device_eval_batch_size=8,            # 평가 배치 크기
    warmup_steps=500,                        # 학습률 워밍업
    weight_decay=0.01,                       # 가중치 감쇠
    logging_steps=10,                        # 로그 간격 (evaluation_strategy, save_strategy)
    eval_strategy="epoch",                   # 에포크마다 평가
    save_strategy="epoch",                   # 에포크마다 저장
    load_best_model_at_end=True,             # 최적 모델 로드
    save_total_limit=3,                      # 최대 저장 파라미터 수
    report_to="none",                        # 결과 리포트
)

In [ ]:
# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=TutorialCollator(tokenizer)
)

In [ ]:
# 학습 실행
train_result = trainer.train()
train_result

In [ ]:
!ls ./results/hf-tutorial

## Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
model = AutoModelForSequenceClassification.from_pretrained("./results/hf-tutorial/checkpoint-39",
                                                           num_labels=2)

In [ ]:
# model 및 tokenizer Hub에 업로드
model.push_to_hub("hf-tutorial")
tokenizer.push_to_hub("hf-tutorial")